# Modelisation - Regression Logistique

Pipeline complet pour la prediction de la gravite des accidents de la route (2024).

**Etapes :** chargement - nettoyage - encodage - normalisation - entrainement - evaluation - validation croisee - analyse des coefficients.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
import warnings

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-darkgrid')


## 1. Chargement et Exploration des Donnees

In [ ]:
base_path = Path.cwd().parent
df = pd.read_csv(base_path / 'data' / 'raw' / 'accidents_2024.csv', sep=';', encoding='latin-1')
df.drop_duplicates(inplace=True)
df.dropna(subset=['Gravite (label)'], inplace=True)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print(f'Donnees chargees : {df.shape[0]} lignes, {df.shape[1]} colonnes')
print(df['Gravite (label)'].value_counts())


## 2. Nettoyage et Preparation des Donnees

In [ ]:
df.drop_duplicates(inplace=True)
df.dropna(subset=['Gravite (label)'], inplace=True)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        df[col].fillna(df[col].mode()[0], inplace=True)

print(f'Lignes apres nettoyage : {df.shape[0]}')
print(df['Gravite (label)'].value_counts())


## 3. Encodage, Decoupage Train/Test et Normalisation

La regression logistique repose sur une optimisation par gradient ; la normalisation accelere la convergence et ameliore la stabilite numerique.

In [ ]:
y = df['Gravite (label)']
X_raw = df.drop(columns=['Num_Acc', 'Departement', 'Gravite (label)'])
X_encoded = pd.get_dummies(X_raw, drop_first=True)

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.20, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Train : {X_train_scaled.shape} | Test : {X_test_scaled.shape}')


## 4. Modelisation avec LogisticRegression

### Strategie de compensation du desequilibre
- **class_weight='balanced'** : les poids de classe sont calcules automatiquement proportionnellement a l'inverse de leur frequence
- **max_iter=1000** : garantit la convergence meme avec beaucoup de features

In [ ]:
model = LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced')
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)
print('Modele entraine.')


## 5. Evaluation du Modele

### Metriques de performance
- **Accuracy** : proportion de predictions correctes
- **Recall** : proportion de vrais positifs detectes *(prioritaire pour la classe Tue)*
- **F1-Score** : moyenne harmonique precision/recall

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

def evaluate_model(y_true, y_pred, label=''):
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    rec  = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1   = f1_score(y_true, y_pred, average='weighted', zero_division=0)
    print(label)
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    return acc, prec, rec, f1

evaluate_model(y_test, y_pred, 'Regression Logistique - Ensemble de test')
print()
print(classification_report(y_test, y_pred))


## 6. Matrice de Confusion

In [ ]:
labels = sorted(y_test.unique())
cm = confusion_matrix(y_test, y_pred, labels=labels)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=labels, yticklabels=labels, ax=ax)
ax.set_title('Matrice de confusion - Regression Logistique', fontweight='bold')
ax.set_xlabel('Predit')
ax.set_ylabel('Reel')
plt.tight_layout()
plt.show()


## 7. Validation Croisee (5-Fold Stratifie)

La validation croisee stratifiee garantit la representation de chaque classe dans chaque pli.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(model, X_train_scaled, y_train,
                          cv=cv, scoring='recall_macro')

print(f'Recall macro - CV 5-fold : {scores.mean():.3f} (+/- {scores.std():.3f})')
print(f'Scores par fold : {scores.round(3)}')


## 8. Analyse des Coefficients - Facteurs Aggravants et Protecteurs

Les coefficients de la regression logistique indiquent l'impact de chaque variable sur la probabilite d'appartenance a la classe Tue : un coefficient positif augmente le risque, un coefficient negatif le reduit.

In [ ]:
index_tue = list(model.classes_).index('Tue')

coef_df = pd.DataFrame({
    'Variable': X_encoded.columns,
    'Coefficient': model.coef_[index_tue]
}).sort_values('Coefficient', ascending=False)

print('--- TOP 5 FACTEURS AGGRAVANTS (augmentent le risque de deces) ---')
print(coef_df.head(5).to_string(index=False))
print()
print('--- TOP 5 FACTEURS PROTECTEURS (diminuent le risque de deces) ---')
print(coef_df.tail(5).to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 6))
top = pd.concat([coef_df.head(5), coef_df.tail(5)])
colors = ['#F44336' if v > 0 else '#2196F3' for v in top['Coefficient']]
ax.barh(top['Variable'], top['Coefficient'], color=colors)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Coefficients classe Tue - Top 5 aggravants / protecteurs', fontweight='bold')
ax.set_xlabel('Coefficient logistique')
plt.tight_layout()
plt.show()


## 9. Resume et Recommandations

In [ ]:
print('=' * 65)
print('RESUME - REGRESSION LOGISTIQUE')
print('=' * 65)
print('  max_iter=1000 | class_weight=balanced | StandardScaler')
print(f'  Recall macro CV : {scores.mean():.3f}')
print()
print('  Avantages : interpretable via coefficients, rapide, bonne baseline.')
print('  Limites   : hypothese de linearite, performances limitees sur donnees complexes.')
